# MS-Human-700 ilk model denetimi
Bu dosya Google Colab'ın Google tarafından sağlanan Python çalışma ortamında çalıştırılmak üzere hazırlandı. İlk amaç modelin derlenmesi, yürüyüş dosyasının okunması ve sentetik örneğin doğrulanmasıdır. Kas zayıflığı deneyinin fizyolojik sonucu henüz üretilmedi.

Colab'da **Dosya > Not defteri yükle** ile bu dosyayı açın. Ardından **Çalışma zamanı > Tümünü çalıştır** seçin. Standart CPU ortamı yeterli. Paketler ve kaynaklar uzak çalışma ortamına indirilir. Google hesabıyla oturum açmanız gerekebilir. Dosya ve kod bu oturumda Colab'da henüz çalıştırılmadı.

Son hücre sonuç ZIP dosyası için bir indirme bağlantısı verir. Kaydederken `D:\projefikri\kas_animasyon` klasörünü seçin. Kaynaklar açık GitHub depolarıdır. Önceki çalışma ile aynı commit ve arşiv sağlama toplamları denetlenir.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess

if not Path('/content').is_dir() or sys.platform != 'linux':
    raise RuntimeError('Bu dosyayı Colab tarafından sağlanan Linux çalışma ortamında açın.')
project = Path('/content/kas_animasyon')
project.mkdir(exist_ok=True)
os.chdir(project)
os.environ['MUJOCO_GL'] = 'egl'
os.environ['MPLCONFIGDIR'] = str(project / '.cache' / 'matplotlib')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                'mujoco>=3.2.6,<4', 'numpy>=1.26,<3', 'scipy>=1.11,<2'], check=True)
print('Uzak çalışma klasörü', project)


In [ ]:
import json
import runpy

scripts = {'fetch_sources.py': '"""Download pinned public sources and record their provenance inside the project."""\n\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path, PurePosixPath\nimport urllib.request\nimport zipfile\n\nROOT = Path(__file__).resolve().parents[1]\nVENDOR = ROOT / \'vendor\'\n\n\ndef request(url):\n    headers = {\'User-Agent\': \'kas-animasyon-research\', \'Accept\': \'application/vnd.github+json\'}\n    with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=120) as response:\n        return response.read()\n\n\ndef unpack_repository(repository, revision, destination):\n    url = f\'https://codeload.github.com/{repository}/zip/{revision}\'\n    payload = request(url)\n    destination.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(io.BytesIO(payload)) as archive:\n        for member in archive.infolist():\n            relative = PurePosixPath(member.filename)\n            components = relative.parts[1:]\n            if not components:\n                continue\n            if any(part in (\'.\', \'..\') or \':\' in part or \'\\\\\' in part for part in components):\n                raise ValueError(f\'Unsafe archive member: {member.filename}\')\n            target = destination.joinpath(*components)\n            if not target.resolve().is_relative_to(destination.resolve()):\n                raise ValueError(\'Archive target escapes destination\')\n            if member.is_dir():\n                target.mkdir(parents=True, exist_ok=True)\n            else:\n                target.parent.mkdir(parents=True, exist_ok=True)\n                target.write_bytes(archive.read(member))\n    return {\'repository\': repository, \'revision\': revision, \'archive_url\': url,\n            \'archive_sha256\': hashlib.sha256(payload).hexdigest(),\n            \'archive_bytes\': len(payload), \'directory\': str(destination.relative_to(ROOT))}\n\n\ndef main():\n    VENDOR.mkdir(exist_ok=True)\n    manifest_path = VENDOR / \'sources.json\'\n    if manifest_path.exists():\n        raise SystemExit(\'Sources already recorded. Inspect vendor/sources.json before downloading again.\')\n    commit = json.loads(request(\'https://api.github.com/repos/LNSGroup/msgym/commits/main\'))\n    revision = commit[\'sha\']\n    tree = json.loads(request(f\'https://api.github.com/repos/LNSGroup/msgym/git/trees/{revision}?recursive=1\'))\n    model = next(item for item in tree[\'tree\'] if item[\'path\'] == \'msgym/MS-Human-700\')\n    if model[\'mode\'] != \'160000\':\n        raise ValueError(\'Expected a pinned model submodule\')\n    records = []\n    for repository, version, folder in (\n        (\'LNSGroup/msgym\', revision, \'msgym\'),\n        (\'LNSGroup/MS-Human-700\', model[\'sha\'], \'MS-Human-700\'),\n    ):\n        print(f\'Downloading {repository} at {version}\', flush=True)\n        records.append(unpack_repository(repository, version, VENDOR / folder))\n    manifest_path.write_text(json.dumps({\'sources\': records}, indent=2) + \'\\n\', encoding=\'utf-8\')\n    print(json.dumps({\'sources\': records}, indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n', 'synthetic_pilot.py': '"""Check the research condition on an explicitly artificial three-actuator system."""\n\nimport json\nfrom itertools import combinations\nfrom pathlib import Path\n\nimport numpy as np\nfrom scipy.optimize import linprog\n\nROOT = Path(__file__).resolve().parents[1]\n\n\ndef solve(failed, dynamic):\n    # This is a two-instant force allocation example, not a physiological muscle model.\n    capacities = np.array([0.45, 0.45, 1.0])\n    capacities[list(failed)] = 0.0\n    target = np.array([0.45, 0.80])\n    rate_limits = np.array([0.45, 0.45, 0.10])\n    equality = np.array([[1, 1, 1, 0, 0, 0], [0, 0, 0, 1, 1, 1]])\n    change = np.hstack((-np.eye(3), np.eye(3)))\n    result = linprog(\n        np.zeros(6), A_eq=equality, b_eq=target,\n        A_ub=np.vstack((change, -change)) if dynamic else None,\n        b_ub=np.tile(rate_limits, 2) if dynamic else None,\n        bounds=[(0.0, cap) for cap in np.tile(capacities, 2)], method=\'highs\',\n    )\n    if result.status not in (0, 2):\n        raise RuntimeError(f\'Solver outcome cannot establish feasibility: {result.message}\')\n    solution = result.x.reshape(2, 3) if result.success else None\n    if solution is not None:\n        if np.max(np.abs(solution.sum(axis=1) - target)) > 1e-8:\n            raise RuntimeError(\'Force balance residual exceeded tolerance\')\n        if dynamic and np.any(np.abs(np.diff(solution, axis=0)[0]) > rate_limits + 1e-8):\n            raise RuntimeError(\'Rate constraint residual exceeded tolerance\')\n    return {\'feasible\': bool(result.success), \'solver_status\': result.status,\n            \'forces\': solution.tolist() if solution is not None else None}\n\n\ndef main():\n    names = (\'A\', \'B\', \'C\')\n    scenarios = {}\n    for size in range(3):\n        for failed in combinations(range(3), size):\n            label = \'+\'.join(names[i] for i in failed) or \'baseline\'\n            scenarios[label] = {\'independent\': solve(failed, False), \'time_linked\': solve(failed, True)}\n    for label in (\'baseline\', \'A\', \'B\'):\n        if not scenarios[label][\'time_linked\'][\'feasible\']:\n            raise RuntimeError(f\'Expected a feasible control scenario: {label}\')\n    if not scenarios[\'A+B\'][\'independent\'][\'feasible\'] or scenarios[\'A+B\'][\'time_linked\'][\'feasible\']:\n        raise RuntimeError(\'Expected a static-feasible, dynamic-infeasible pair\')\n    # Independent check. With A+B absent, C must increase by .35; its bound is .10.\n    report = {\n        \'kind\': \'synthetic numerical check\', \'physiological_evidence\': False,\n        \'scope\': \'two fixed time instants, arbitrary force units, complete actuator removal\',\n        \'independent_infeasibility_check\': {\'required_change_C\': 0.35, \'allowed_change_C\': 0.10},\n        \'scenarios\': scenarios,\n    }\n    output = ROOT / \'results\' / \'synthetic\'\n    output.mkdir(parents=True, exist_ok=True)\n    (output / \'pilot.json\').write_text(json.dumps(report, indent=2) + \'\\n\', encoding=\'utf-8\')\n    for name, scenario in scenarios.items():\n        print(f\'{name:8} independent={scenario["independent"]["feasible"]} \'\n              f\'time_linked={scenario["time_linked"]["feasible"]}\')\n    print(\'Synthetic checks passed. Physiological evidence requires the real-model experiment.\')\n\n\nif __name__ == \'__main__\':\n    main()\n', 'audit_model.py': '"""Inspect downloaded XML and numeric motion data, then attempt MuJoCo compilation."""\n\nimport csv\nimport hashlib\nimport json\nfrom pathlib import Path\nimport xml.etree.ElementTree as ET\n\nimport numpy as np\n\nROOT = Path(__file__).resolve().parents[1]\nMODELS = ROOT / \'vendor\' / \'MS-Human-700\'\nMOTION = ROOT / \'vendor\' / \'msgym\' / \'msgym\' / \'motion_data\' / \'walking_gait.npz\'\nOUTPUT = ROOT / \'results\' / \'model_audit\'\n\n\ndef expand_xml(path, active=()):\n    path = path.resolve()\n    if not path.is_relative_to(MODELS.resolve()) or path in active:\n        raise ValueError(f\'Invalid or cyclic include: {path}\')\n    root = ET.parse(path).getroot()\n    for parent in list(root.iter()):\n        for child in list(parent):\n            if child.tag == \'include\':\n                # MuJoCo resolves included files against the top-level model directory.\n                included = expand_xml(MODELS / child.attrib[\'file\'], (*active, path))\n                index = list(parent).index(child)\n                parent.remove(child)\n                for offset, item in enumerate(list(included)):\n                    parent.insert(index + offset, item)\n    return root\n\n\ndef inspect_xml(filename):\n    tree = expand_xml(MODELS / filename)\n    joints = list(tree.findall(\'.//worldbody//joint\'))\n    if tree.findall(\'.//worldbody//freejoint\') or any(\n        joint.get(\'type\', \'hinge\') not in (\'hinge\', \'slide\') for joint in joints\n    ):\n        raise ValueError(\'Scalar joint coordinate assumption must be revised\')\n    actuators = list(tree.findall(\'./actuator/*\'))\n    return {\n        \'filename\': filename,\n        \'source_sha256\': hashlib.sha256((MODELS / filename).read_bytes()).hexdigest(),\n        \'xml_joint_count\': len(joints),\n        \'xml_body_count_excluding_world\': len(tree.findall(\'.//worldbody//body\')),\n        \'xml_actuator_count\': len(actuators),\n        \'joint_names\': [joint.get(\'name\') for joint in joints],\n        \'actuator_names\': [actuator.get(\'name\') for actuator in actuators],\n        \'joint_equalities\': [dict(item.attrib) for item in tree.findall(\'./equality/joint\')],\n        \'keyframe_coordinate_count\': len(tree.find(\'./keyframe/key\').get(\'qpos\').split()),\n    }\n\n\ndef main():\n    OUTPUT.mkdir(parents=True, exist_ok=True)\n    report = {\'stage\': \'source and numeric data audit\', \'motion\': {}, \'models\': []}\n    with np.load(MOTION, allow_pickle=False) as motion:\n        report[\'motion\'][\'file\'] = str(MOTION.relative_to(ROOT))\n        report[\'motion\'][\'sha256\'] = hashlib.sha256(MOTION.read_bytes()).hexdigest()\n        report[\'motion\'][\'fields\'] = {}\n        for key in motion.files:\n            try:\n                values = motion[key]\n            except ValueError as error:\n                if \'Object arrays cannot be loaded\' not in str(error):\n                    raise\n                report[\'motion\'][\'fields\'][key] = {\'loaded\': False, \'reason\': \'object metadata requires pickle\'}\n                continue\n            if not np.issubdtype(values.dtype, np.number):\n                raise ValueError(f\'Unexpected nonnumeric field: {key}\')\n            if not np.isfinite(values).all():\n                raise ValueError(f\'Nonfinite motion values: {key}\')\n            record = {\'loaded\': True, \'shape\': list(values.shape), \'dtype\': str(values.dtype), \'finite\': True}\n            if values.ndim == 0:\n                record[\'value\'] = values.item()\n            report[\'motion\'][\'fields\'][key] = record\n        qpos = motion[\'qpos_traj\']\n        xpos = motion[\'xpos_traj\']\n        rate = float(motion[\'framerate\'])\n        if qpos.ndim != 2 or rate <= 0 or qpos.shape[0] < 3:\n            raise ValueError(\'Invalid motion dimensions or rate\')\n        report[\'motion\'][\'frame_count\'] = len(qpos)\n        report[\'motion\'][\'sample_rate_hz\'] = rate\n        report[\'motion\'][\'sample_span_s\'] = (len(qpos) - 1) / rate\n        report[\'motion\'][\'repository_cycle_duration_s\'] = len(qpos) / rate\n        report[\'motion\'][\'external_loads_present\'] = any(\n            token in key.lower() for key in motion.files for token in (\'grf\', \'contact_force\', \'wrench\', \'torque\')\n        )\n        with (OUTPUT / \'reference_coordinates.csv\').open(\'w\', newline=\'\', encoding=\'utf-8\') as handle:\n            writer = csv.writer(handle)\n            # Use source indices until correspondence with the compiled model is established.\n            writer.writerow([\'time_s\', *(f\'source_qpos_{i}\' for i in range(qpos.shape[1]))])\n            for index, row in enumerate(qpos):\n                writer.writerow([index / rate, *row])\n\n    for filename in (\'MS-Human-700.xml\', \'MS-Human-700-Locomotion.xml\'):\n        report[\'models\'].append(inspect_xml(filename))\n    full, legs = report[\'models\']\n    if full[\'xml_joint_count\'] != qpos.shape[1]:\n        raise ValueError(\'Reference and full model coordinate counts differ\')\n    if xpos.shape != (len(qpos), full[\'xml_body_count_excluding_world\'] + 1, 3):\n        raise ValueError(\'Reference body-position shape and XML body count differ\')\n    if [model[\'xml_actuator_count\'] for model in report[\'models\']] != [700, 100]:\n        raise ValueError(\'Unexpected actuator counts in pinned model sources\')\n    if any(model[\'xml_joint_count\'] != model[\'keyframe_coordinate_count\'] for model in report[\'models\']):\n        raise ValueError(\'Keyframe coordinate count mismatch\')\n    if not set(legs[\'joint_names\']).issubset(full[\'joint_names\']):\n        raise ValueError(\'Leg joint names are not a subset of full model\')\n    report[\'coordinate_mapping\'] = {\n        \'source_column_order_validated\': False,\n        \'xml_names_compatible\': True,\n        \'legs_to_full_xml_joint_indices\': [full[\'joint_names\'].index(name) for name in legs[\'joint_names\']],\n        \'note\': \'XML names and dimensions match; compiled kinematic correspondence remains pending.\',\n    }\n    report[\'mujoco\'] = {\'imported\': False, \'compiled\': False}\n    try:\n        import mujoco\n    except ImportError as error:\n        report[\'mujoco\'][\'error\'] = str(error)\n        report[\'mujoco\'][\'cause\'] = str(error.__cause__) if error.__cause__ else None\n    else:\n        report[\'mujoco\'][\'imported\'] = True\n        report[\'mujoco\'][\'version\'] = mujoco.__version__\n        for model_record in report[\'models\']:\n            model = mujoco.MjModel.from_xml_path(str(MODELS / model_record[\'filename\']))\n            data = mujoco.MjData(model)\n            mujoco.mj_resetDataKeyframe(model, data, 0)\n            mujoco.mj_forward(model, data)\n            if not np.isfinite(data.qacc).all():\n                raise ValueError(\'Nonfinite initialization accelerations\')\n            model_record[\'compiled\'] = {\'nq\': model.nq, \'nv\': model.nv, \'nu\': model.nu, \'nbody\': model.nbody}\n        report[\'mujoco\'][\'compiled\'] = True\n    (OUTPUT / \'audit.json\').write_text(json.dumps(report, indent=2) + \'\\n\', encoding=\'utf-8\')\n    print(f\'Motion: {len(qpos)} frames, {rate:g} Hz, {qpos.shape[1]} source coordinates\')\n    for model in report[\'models\']:\n        print(f\'{model["filename"]}: XML joints={model["xml_joint_count"]}, actuators={model["xml_actuator_count"]}\')\n    print(f\'MuJoCo compiled: {report["mujoco"]["compiled"]}\')\n    if \'error\' in report[\'mujoco\']:\n        print(\'MuJoCo import blocker:\', report[\'mujoco\'])\n\n\nif __name__ == \'__main__\':\n    main()\n'}
(project / 'scripts').mkdir(exist_ok=True)
for name, source in scripts.items():
    (project / 'scripts' / name).write_text(source, encoding='utf-8')
helpers = runpy.run_path(str(project / 'scripts' / 'fetch_sources.py'))
pinned = [{'repository': 'LNSGroup/msgym', 'revision': 'ad3aac166dab3577d7d16b4a03faccef94c7aac7', 'archive_url': 'https://codeload.github.com/LNSGroup/msgym/zip/ad3aac166dab3577d7d16b4a03faccef94c7aac7', 'archive_sha256': '37dad71e1e79c3c53169ae4520134a9aedde8710afc4f4bf15a3491e8ee56c76', 'archive_bytes': 20540062, 'directory': 'vendor\\msgym'}, {'repository': 'LNSGroup/MS-Human-700', 'revision': '2d686957aefd5739cf4d2859a6acfd94c8f84400', 'archive_url': 'https://codeload.github.com/LNSGroup/MS-Human-700/zip/2d686957aefd5739cf4d2859a6acfd94c8f84400', 'archive_sha256': '65975d132606149e7707c8ce0b6015c33e2d6fd50d15f00e416f961eb47327bc', 'archive_bytes': 30783005, 'directory': 'vendor\\MS-Human-700'}]
manifest_path = project / 'vendor' / 'sources.json'
if not manifest_path.exists():
    records = []
    for item in pinned:
        folder = item['repository'].split('/')[-1]
        record = helpers['unpack_repository'](item['repository'], item['revision'], project / 'vendor' / folder)
        if record['archive_sha256'] != item['archive_sha256']:
            raise RuntimeError('Kaynak arşiv sağlama toplamı uyuşmuyor. Sonuçları kullanmadan önce inceleyin.')
        records.append(record)
    manifest_path.write_text(json.dumps({'sources': records}, indent=2), encoding='utf-8')
print(manifest_path.read_text())


In [ ]:
runpy.run_path(str(project / 'scripts' / 'synthetic_pilot.py'), run_name='__main__')
runpy.run_path(str(project / 'scripts' / 'audit_model.py'), run_name='__main__')
audit = json.loads((project / 'results' / 'model_audit' / 'audit.json').read_text())
if not audit['mujoco']['compiled']:
    raise RuntimeError(audit['mujoco'])
print('İki gerçek model MuJoCo ile derlendi. Referansın boyutları ve sayısal alanları kontrol edildi.')
print('Koordinatların fiziksel eşlemesi ve dış yükler sonraki adımda doğrulanacak.')


Model derlemesi başarılıysa başlangıç denetimi tamamlandı. Hazır hareket kaydı dış yükleri içermediği için bu sonuç kas zayıflığı deneyini tamamlamaz. Başarılı veya hatalı çıktıyı proje sohbetinde paylaşabilirsiniz.


In [ ]:
import importlib.metadata
import shutil
from google.colab import files

versions = {name: importlib.metadata.version(name) for name in ('mujoco', 'numpy', 'scipy')}
(project / 'results' / 'colab_versions.json').write_text(json.dumps(versions, indent=2), encoding='utf-8')
archive = shutil.make_archive(str(project / 'colab_sonuclari'), 'zip', root_dir=project / 'results')
files.download(archive)
